# Run this cell first

In [ ]:
# this code enables the automated feedback. If you remove this, you won't get any feedback
# so don't delete this cell!
try:
  import AutoFeedback
except (ModuleNotFoundError, ImportError):
  %pip install AutoFeedback
  import AutoFeedback

try:
  from testsrc import test_main
except (ModuleNotFoundError, ImportError):
  %pip install "git+https://github.com/autofeedback-exercises/exercises.git#subdirectory=MTH2100/numerical_analysis/ode_euler_rk4"
  from testsrc import test_main

def runtest(tlist):
  import unittest
  from contextlib import redirect_stderr
  from os import devnull
  with redirect_stderr(open(devnull, 'w')):
    suite = unittest.TestSuite()
    for tname in tlist:
      suite.addTest(eval(f"test_main.UnitTests.{tname}"))
    runner = unittest.TextTestRunner()
    try:
      runner.run(suite)
    except (AssertionError, ImportError):
      pass

# 4. Numerical Solution of ODEs: Euler and Runge–Kutta Methods

## Why this matters

Almost every dynamical model — whether in physics, biology, chemistry, or economics —
is described by a system of ordinary differential equations (ODEs). The reason for this is simple: the way that things change – $\frac{df}{dx}$ –is usually related to the state those things are in  – $f(x)$.  While analytical solutions
exist only for special special cases, numerical ODE solvers let us simulate any model we can write
down.

This notebook builds ODE solvers from scratch, starting with the simplest (Euler's method)
and progressing to the workhorse of scientific computing (RK4). As with other numerical analysis techniques we have looked at so far, you usually don't need to write your own methods because there are libraries of functions to do the job. However, understanding these methods
from first principles will help you choose appropriate solvers and understand when and why
they go wrong. 

**Learning objectives**
- Formulate an initial value problem (IVP) and understand what a numerical solver does
- Implement forward Euler's method and understand its limitations
- Understand why energy drift is a fundamental problem for symplectic systems
- Implement classical fourth-order Runge–Kutta (RK4)
- Compare the convergence and stability of Euler and RK4

<div style="background:#e8f4f8;border-left:4px solid #2196F3;padding:16px;margin:16px 0;border-radius:4px;">

**📹 VIDEO: Numerical ODE Solvers: Euler and Runge–Kutta**

*What this video covers:* The initial value problem; slope fields and what integration means geometrically; deriving Euler's method from the Taylor series; the local truncation error and global error; deriving RK4 as a weighted average of four slopes; comparing convergence orders.

*(Embed the video here using an `<iframe>` or `IPython.display.Video`)*
</div>

## 0. A helpful way to think about calculus numerically

Before we get into methods, a quick word about calculus. When you learned how to differentiate functions, you probably learned rules for determining $\frac{dy}{dx}$. The quantity $\frac{dy}{dx}$ thus takes on some sort of mythical quantity, and we lose all sense of what it actually means or where it comes from. So two things to remember

### 1. $\frac{dy}{dx}$ means 'the rate at which y changes with respect to x'. 

I know that sounds trivial – that's the definition of a derivative – but really try to stop and think about that in the context of whatever problem you're solving. For time-based problems, it's a bit easier – it's an actual rate. The derivative of position with respect to time is the rate of change of position, which we call velocity. The derivative of a bacteria colony's size with respect to time is its growth rate. For spatial problems, it's sometimes less obvious, but you can usually imagine changing something 'with time' and that can help. For instance, if you have a piece of plastic that curve, forming a curve in 2-D, the derivative of the y coordinate with respect to the x coordinate tells you how 'quickly' the y coordinate is changing as you move in the x-direction. I always picture this as a little stick figure trying to move up or down a hill. A large $\frac{dy}{dx}$ means that the hill is really steep, a negative $\frac{dy}{dx}$ means that the hill is sloping downwards etc. 

### $dy$ and $dx$ are special, but $\Delta y$ and $\Delta x$ are just numbers. 

This might be the most important concept in the entire course. The _actual_ definition of the derivative (mathematically, rather than conceptually) is 

$$\dfrac{df}{dx} = \lim _{\Delta x \to 0} \dfrac{f(x + \Delta x) - f(x)}{\Delta x}$$

A picture might be instructive

<center><img src="./gradient.png" alt="gradients.png" width="500"></img></center>


In the image on the left, the gradient of the red curve at $x$ is just the gradient of the green tangent line. The gradient of the blue line doesn't look like it's anywhere close to the gradient of the green line, but at least it's easy to calculate: it's just the change in the value of the function, $\Delta f = f(x+\Delta x) - f(x)$, divided by the change in $x$, $\Delta x$ (i.e. rise over run). Clearly if we make $\Delta x$ smaller, as we have done in the image on the right, the gradient of the blue line does get closer to the gradient of the green line. **The whole idea of calculus is that we can make the gap smaller and smaller until the two things are the same**. In other words, 'in the limit that $\Delta x$ tends to zero', the gradient is just $\dfrac{\Delta f}{\Delta x}.$

Now for most applications we don't care about things being _exactly_ correct (which requires the 'limit'), and we can't divide things by zero anyway. We just want numbers that are close to correct. Thus instead of trying to calculate the mythical $\frac{dy}{dx}$ we can simply calculate the _numbers_ $\Delta y$ and $\Delta x$, and use those. 

This also has the advantage that we can move those numbers around, in ways that we can't always do with the infinitesimals $dy$ and $dx$. For instance, if you have a differential equation 

$$\dfrac{dy}{dx} = y$$

to solve it you have to rearrange by dividing through by $y$, and then you have to _integrate_ (ugh) both sides with respect to $x$ to find a solution. 

If we instead have 

$$\dfrac{\Delta y}{\Delta x} = y$$

Then you can calculate the _change in y_ – which is what $\Delta y$ means – simply by multiplying both sides by the change in $x$:

$$ \Delta y = y \Delta x$$
Most methods for solving differential equations numerically amount to working with the deltas like that, and (ultimately) with fancier and fancier ways of computing $\Delta y$. 

---

## 1. Euler's Method

With that initial section done, we can immediately apply it to solve a really simple differential equation: 

$$ \dfrac{dy}{dt} = y(t) .$$

You should know that the solution of this equation is just $y = y_0e^t$ where $y_0 = y(t=0)$. (The differential equation says 'the rate of change of the function is equal to the function': that's the _definition_ of the exponential function). But with our numerical analysis hat on, we can replace the differential equation with 

$$\dfrac{\Delta y}{\Delta t} \approx y(t)$$

So, if I know what the value of the function is at some point $t$, then I can work out by how much the function changes

$$ \Delta y \approx y(t) \Delta t, $$

and thereby what the value of the function 'will be' at some point $\Delta t$ 'later'. 


$$ y(t+\Delta t) = y(t) + \Delta y = y(t) + y(t)\Delta t $$

We do know what the value of the function is at one point: at $t=0$ we have the 'initial value', $y_0$. Thus we use our formula to compute the value of $y$ at $t = \Delta t$, then from that value we can compute the next value at $t=2\Delta t$ and so on. Let's do that for $y_0 = 1$ and, and look at how our approximate solution compares to the exact, exponential function. This approach is called the Euler method.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def delta_y(y, delta_t):
    return y * delta_t

delta_t = 0.1
t = 0
y = 1

t_array = [t]
y_array = [y] 

N_steps = int(2/delta_t) # how many steps to cover a the range from t=0 to t=2

for _ in range(N_steps):
    y = y + delta_y(y, delta_t)
    t = t + delta_t 
    y_array.append(y)
    t_array.append(t)


plt.plot(t_array, y_array, 'ro', label='Euler Method')
plt.plot(t_array, np.exp(t_array), 'bo', label='Exact')

plt.xlabel('t')
plt.ylabel('y(t)')

plt.legend(['Euler method', 'Exact'])



What you should notice is that the approximation is pretty good, but it gets worse as we go to larger and larger values of $t$. This is because at each succesive point we are accumulating small errors, which add up as we go along (like taking a photocopy of a photocopy). We can fix this by making $\Delta t$ smaller (try it above), but then you should notice that the number of points plotted becomes larger too, i.e. we have to take more steps to cover the same period of time. In the limit that $\Delta t$ goes to zero, we'll get something exact – woohoo! – but unfortunately, we'll have to compute infinitely many little steps – awwww. This, by the way, is the essence of [Zeno's paradoxes](https://en.wikipedia.org/wiki/Zeno%27s_paradoxes).

---


While Euler's method the way we've described it is a great way to understand the basic ideas of numerical calculus, it is helpful to take a slightly more formal approach to its derivation, as that will help us see how to extend to other methods. 

Formally, we start with the Taylor series:

$$y(t+ \Delta t) = y(t) +  \Delta t y'(t) + \frac{(\Delta t)^2}{2!}y''(t) + \cdots$$

If $\Delta t$ is sufficiently small, then $(\Delta t)^2$ is negligible. Thus, dropping all terms of order $(\Delta t)^2$ and above, we obtain

$$y(t+ \Delta t) = y(t) +  \Delta t \frac{dy}{dt}$$

You can see that for the specific, prototype differential equation we've been working with, $\frac{dy}{dt}=y$, we could simply replace the derivative on the RHS of this expression and recover exactly what we had before. 

The reason it's useful to do things this way is to recognise that we have made the arbitrary decision to neglect the terms of $O(\Delta t^2)$, but we could, in principle retain higher order terms. Neglecting terms of order $O(\Delta t)^2$ means that the error is proportional to $(\Delta t)^2$ (and remember, $\Delta t \ll 1$ so squaring it makes the error small!) Higher order methods will have even smaller errors $^*$. 

$^*$ Note that there is a difference between what's called the Local Truncation Error (LTE) and the Global Truncation Error: The LTE for the Euler method is proportional to $h^2$, but the global error, which accumulates all those small errors, behaves more like $h$. That means Euler is actually a pretty inaccurate method). 

<div style="background:#f7d6e3;border-left:4px solid #f32121;padding:16px;margin:16px 0;border-radius:4px;width:100%;box-sizing:border-box;overflow-wrap:break-word;">

# Exercise

Implement the Euler method to solve the differential equation $$ \frac{df}{dx} = \cos(f(x)),$$ with the initial condition, $f(x=0) = 0$, over the range $x=0$ to $x=10$.

You can use the approach above as a template:

1. Implement a function called `delta_f` which takes `f` and `delta_x` as arguments and returns the value of the 'change in f'. 
2. Define the initial values of `f` and `x`
3. Define `delta_x` to be a sufficiently small value
4. Calculate the number of steps required to cover the range (remember this has to be an integer, i.e. a whole number of steps)
5. Iterate through the required number of steps calculating the updated value of `f` and `x` each time and storing those values in arrays/lists called `f_array` and `x_array`
6. Plot the solution

</div>

In [ ]:
# your code goes here



# This code is required for the autofeedback- don't delete it!
fighand = plt.gca()

In [ ]:
runtest(['test_delta_f', 'test_arrays', 'test_plot'])

## 2. Second order differential equations

Methods like the Euler method (and others that we shall see, like RK4) work for first order differential equations, so you might think that we need to find alternative methods for the whole class of problems in nature that require higher order differential equations. However, we can always recast higher order, linear differential equations as _systems_ of first order differential equations, and use the same methods to solve those systems. 

For instance, take the prototypical second-order problem, the simple harmonic oscillator. This is governed by the equation

$$ \dfrac{d^2x}{dt^2} = -\omega ^2 x(t) .$$

To cast this as a system of first order differential equations, we simply introduce a new quantity

$$v = \dfrac{dx}{dt}. \quad (*)$$

Substituting this into the second order equation above we obtain

$$ \dfrac{dv}{dt} = -\omega ^2 x(t). \quad(§)$$

Together, Equations (*) and (§) form a _coupled system of first order equations_ and they can be solved to determine the solution. The advantage with the way we've set this up is that $v$ is of course a useful and meaningful quantity: it is the velocity of the harmonic oscillator. This gives another advantage to setting up the problem this way: the solution gives us direct access to a physically meaningful quantity. 

Practically, what we do is change the function that updates the value of our dependent variable (previously we called this `delta_f`) so that it updates both dependent variables in our coupled system (here, `x` and `v`). The  implementation of this can get a bit fiddly, because there are various ways you can store the calculated quantities. What I have done below is to store everything in the same format that is output by many of the standard ODE solvers (we'll see one later). 

In [ ]:
omega = 1

def euler(f, t0, y0, t_end, h):
    '''
    Integrate dy/dt = f(t, y) from t0 to t_end using Euler
    '''
    y0 = np.atleast_1d(np.array(y0, dtype=float))
    t  = np.arange(t0, t_end + h/2, h)
    y  = np.zeros((len(t), len(y0)))
    y[0] = y0

    for i in range(len(t) - 1):
        ti, yi = t[i], y[i]

        # calculate [dx/dt, dv/dt]
        delta = f(ti, yi)

        # [x', v'] = [x, v]  + [dx, dv]
        y[i+1] = yi + h * np.asarray(delta)

    return t, y.T


def sho(t, state):
# Simple harmonic oscillator: [dx/dt, dv/dt]
    x, v = state
    return [v, -omega**2 * x]
    
initial_condition = [1, 0] # this means the initial position is 1 and the initial velocity is 0

t, s = euler(sho, 0, initial_condition, 10, 0.01)
plt.plot(t, s[0], label='position')
plt.plot(t, s[1], label='velocity')
plt.xlabel('time (s)')
plt.ylabel('position (m)/velocity (m/s)')
plt.legend()
plt.show()

You may have seen the solution of ordinary differential equations like this before, and you might have learned the method of solution too (integrating factors, trial solutions etc). The advantage with a numerical method is that once you have set up the solver, you can mess about with the system and get updated results instantly. With analytical methods that is not always possible (often you have to rederive the solution, for instance). Try changing the initial conditions in the code above. We had previously that the initial condition was 
```python 
initial_condition = [1, 0]
```
which corresponds to releasing the oscillator from rest at a maximum displacement of 1 (i.e. initial position is `1` and initial velocity is `0`). You can change these values to anything you want- the standard one is the give the oscillator a 'kick' in its equilibrium position (`initial_condition = [0, 1]` for instance). 

---

## Phase
Because we have both the position and velocity, let's take a moment to introduce another way of thinking about the solution of these equations. If you've taken classical mechanics you might have seen this already. It's called the 'phase picture', and it basically means that instead of plotting both position and velocity as a function of time, we plot velocity as a function of position. If you think for a minute about what that should look like you can see that our oscillator starts at a position of $+1$ with a velocity of $0$, when it reaches a position of $0$ it has a velocity of $-1$, when it has a position of $-1$ the velocity is $0$ and finally, when it is at $x=0$ again the velocity is now $+1$. Think a little more deeply and you'll realise that $x = \cos(t)$, $y=-\sin(t)$ is the parametric equation of a circle, so the phase diagram should look like a circle. 

In [ ]:
plt.plot(s[0], s[1])
plt.ylabel('v')
plt.xlabel('x')
plt.axis('equal')
plt.show()

OK: it's roughly circular, but something is going wrong. In the next section we'll investigate to see what is happening.

<div style="background:#f7d6e3;border-left:4px solid #f32121;padding:16px;margin:16px 0;border-radius:4px;width:100%;box-sizing:border-box;overflow-wrap:break-word;">

# Exercise

Use the `euler` function we defined in the last box to solve the inhomogeneous second order differential equation

$$ \dfrac{d^2y}{dx^2}  -\dfrac{dy}{dx} -6y = 3x+2 $$

with the initial conditions $y(x=0) = 0.3$ and $\frac{dy}{dx}\vert_{x=0} = 0$.

1. (By hand) cast this as a system of two, first order equations by introducing a new variable $v=\dfrac{dy}{dx}$. 
2. Implement a new function `update_inhomo` which takes two arguments `x` and `state` where `state` is the 'current' value of `y` and `v`, and returns a list containing $dy/dx$ and $dv/dx$.
3. Call the Euler function with the appropriate arguments to compute the solution between $x=0$ and $x=1$ using a step size of $0.01$.
4. Plot the solution (i.e the value of $y$ as a function $x$)
</div>

In [ ]:
# your code goes here



# This code is required for the autofeedback- don't delete it!
fighand = plt.gca()

In [ ]:
runtest(['test_inhomo', 'test_inhomo_plot'])

## 3. Energy Drift — Why Euler Fails for Oscillatory Systems

For a **conservative** physical system like the simple harmonic oscillator (SHO), the
total mechanical energy $E = \tfrac{1}{2}v^2 + \tfrac{1}{2}\omega^2 x^2$ is conserved.
A good numerical integrator should preserve this.

**Euler's method does not.** Because it is not *symplectic* (it does not respect the
geometric structure of Hamiltonian mechanics), it systematically adds energy to the system
over time. This makes Euler's method unsuitable for long-time simulation of oscillatory or
planetary systems.

In [ ]:
# ── Energy drift in Euler's method on the simple harmonic oscillator ──────────
# Equation of motion: d^2x/dt^2 = -omega^2 * x
# As a system: dx/dt = v,  dv/dt = -omega^2 * x
# State vector: [x, v]

omega = 1.0   # angular frequency

def sho(t, state):
    # Simple harmonic oscillator: [dx/dt, dv/dt]
    x, v = state
    return [v, -omega**2 * x]

# Initial conditions: x(0)=1, v(0)=0  => exact solution: x(t)=cos(t), v(t)=-sin(t)
y0_sho = [1.0, 0.0]
T_total = 50.0   # integrate for 50 time units (many oscillations)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

colors = {'h=0.5': ('r', 0.5), 'h=0.1': ('b', 0.1), 'h=0.05': ('g', 0.05)}

for (label, (col, h)) in colors.items():
    t_eu, y_eu = euler(sho, 0, y0_sho, T_total, h)
    x_eu, v_eu = y_eu[0], y_eu[1]
    energy_eu  = 0.5 * v_eu**2 + 0.5 * omega**2 * x_eu**2

    # Time series position
    axes[0, 0].plot(t_eu, x_eu, color=col, lw=1, alpha=0.8, label=label)

    # Time series velocity
    axes[0, 1].plot(t_eu, v_eu, color=col, lw=1, alpha=0.8, label=label)

    # Phase portrait
    axes[1, 0].plot(x_eu, v_eu, color=col, lw=1, alpha=0.8, label=label)

    # Energy vs time
    axes[1, 1].plot(t_eu, energy_eu, color=col, lw=1, alpha=0.8, label=label)

# Exact solution
t_exact = np.linspace(0, T_total, 2000)
x_exact = np.cos(omega * t_exact)
v_exact = -omega * np.sin(omega * t_exact)
E_exact = 0.5 * v_exact**2 + 0.5 * omega**2 * x_exact**2

axes[0, 0].plot(t_exact, x_exact, 'k-', lw=2, label='Exact', zorder=5)
axes[0, 0].set_title("Position $x(t)$", fontsize=12)
axes[0, 0].set_xlabel('$t$'); axes[0, 0].set_ylabel('$x(t)$')
axes[0, 0].legend(fontsize=9); axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(t_exact, v_exact, 'k-', lw=2, label='Exact', zorder=5)
axes[0, 1].set_title("Velocity $v(t)$", fontsize=12)
axes[0, 1].set_xlabel('$t$'); axes[0, 0].set_ylabel('$v(t)$')
axes[0, 1].legend(fontsize=9); axes[0, 0].grid(alpha=0.3)

axes[1, 0].plot(x_exact, v_exact, 'k-', lw=2, label='Exact (circle)', zorder=5)
axes[1, 0].set_title("Phase portrait $(x, v)$\nExact = circle; Euler = spiral out", fontsize=12)
axes[1, 0].set_xlabel('$x$'); axes[0, 1].set_ylabel('$v$')
axes[1, 0].legend(fontsize=9); axes[0, 1].grid(alpha=0.3)

axes[1, 1].axhline(0.5, color='k', ls='--', lw=1.5, label='True energy = 0.5')
axes[1, 1].set_title("Energy $E(t) = \\frac{1}{2}v^2 + \\frac{1}{2}\\omega^2 x^2$", fontsize=12)
axes[1, 1].set_xlabel('$t$'); axes[1, 0].set_ylabel('$E(t)$')
axes[1, 1].legend(fontsize=9); axes[1, 0].grid(alpha=0.3)
axes[1, 1].set_ylim(0, 3)

plt.suptitle("Euler's method on the simple harmonic oscillator — Energy Drift", fontsize=13)
plt.tight_layout(); plt.show()

Importantly, this energy drift is NOT a consequenc of using a finite step:
it is a structural property of Euler's method (it is not _symplectic_). 
No matter how small the step size is, Euler will eventually spiral outward on an SHO.

RK4 (next section) dramatically reduces this by being markedly more accurate—  but symplectic integrators like Störmer–Verlet are the true fix for conservative systems.

## 4. Classical Runge–Kutta (RK4)

The idea behind Runge–Kutta methods is to evaluate $f(t, y)$ at several intermediate
points within each time step and combine them to get a higher-order estimate of $y(t+h)$.

Remember that the basic underlying principle here is that we are using straight lines to approximate the gradient to our curve. In Euler, we used a single straight line. 
**Classical RK4** uses four slope evaluations per step:

$$\begin{aligned}
k_1 &= f(t_n,\; y_n)\\
k_2 &= f\!\left(t_n + \tfrac{h}{2},\; y_n + \tfrac{h}{2} k_1\right)\\
k_3 &= f\!\left(t_n + \tfrac{h}{2},\; y_n + \tfrac{h}{2} k_2\right)\\
k_4 &= f\!\left(t_n + h,\; y_n + h\, k_3\right)
\end{aligned}$$

$$\boxed{y_{n+1} = y_n + \frac{h}{6}\left(k_1 + 2k_2 + 2k_3 + k_4\right)}$$

The values $k_1 \ldots k_4$ are the gradients of four different slopes related to the curve, and the final value for the slope of the curve is a weighted average of these four numbers.

<p><center><img src="https://upload.wikimedia.org/wikipedia/commons/thumb/7/7e/Runge-Kutta_slopes.svg/500px-Runge-Kutta_slopes.svg.png?utm_source=en.wikipedia.org&amp;utm_campaign=imageinfo&amp;utm_content=thumbnail" alt="Runge-Kutta slopes.svg" height="500" width="500"><br>By <a href="//commons.wikimedia.org/w/index.php?title=User:HilberTraum&amp;action=edit&amp;redlink=1" class="new" title="User:HilberTraum (page does not exist)">HilberTraum</a> - <span class="int-own-work" lang="en">Own work</span>, <a href="https://creativecommons.org/licenses/by-sa/4.0" title="Creative Commons Attribution-Share Alike 4.0">CC BY-SA 4.0</a>, <a href="https://commons.wikimedia.org/w/index.php?curid=64366870">Link</a></center></p>

You can work through the derivation to find that the **Local truncation error** is $O(h^5)$ which is already much more accurate than Euler. But the real advantage for RK4 is that the **Global error** is of order $O(h^4)$ — halving $h$ reduces error by a factor of 16 which is a huge advantage over Euler (where halving $h$ merely halved the error).

The codebox below implements the RK4 and then tests it on a simple problem with a known solution, namely expoential decay:

$$\dfrac{dy}{dx} = -y$$

with intial condition $y(0)=1$ which has the solution

$$y(x) = e^{-x}.$$

In [ ]:
# ════════════════════════════════════════════════════════════════
# CLASSICAL RK4
# ════════════════════════════════════════════════════════════════

def rk4(f, t0, y0, t_end, h):
    '''
    Integrate dy/dt = f(t, y) from t0 to t_end using classical RK4.
    '''
    y0 = np.atleast_1d(np.array(y0, dtype=float))
    t  = np.arange(t0, t_end + h/2, h)
    y  = np.zeros((len(t), len(y0)))
    y[0] = y0

    for i in range(len(t) - 1):
        ti, yi = t[i], y[i]

        # Four slope evaluations
        k1 = np.asarray(f(ti,         yi))
        k2 = np.asarray(f(ti + h/2,   yi + h/2 * k1))
        k3 = np.asarray(f(ti + h/2,   yi + h/2 * k2))
        k4 = np.asarray(f(ti + h,     yi + h   * k3))

        # Weighted average: k1 and k4 get weight 1; k2 and k3 get weight 2
        y[i+1] = yi + (h / 6) * (k1 + 2*k2 + 2*k3 + k4)

    return t, y.T

# ── Convergence comparison: Euler vs RK4 on exponential decay ───────────────
print("= Convergence study: dy/dt = -y, y(0)=1, error at t=5 =")
print()
print(f"{'h':>8} | {'Euler error':>14} | {'RK4 error':>14}")
print("-" * 42)

def f_decay(t, y):
    '''update function for the exponential decay problem'''
    return -y

h_vals   = [0.5, 0.25, 0.1, 0.05, 0.025, 0.01]
true_val = np.exp(-5)
for h in h_vals:
    _, y_eu = euler(f_decay, 0, [1.0], 5, h)
    _, y_rk = rk4(f_decay,  0, [1.0], 5, h)
    eu_err  = abs(y_eu[0, -1] - true_val)
    rk_err  = abs(y_rk[0, -1] - true_val)
    print(f"{h:>8.3f} | {eu_err:>14.4e} | {rk_err:>14.4e}")

In [ ]:
# ── RK4 vs Euler on the simple harmonic oscillator ───────────────────────────
h = 0.2  # moderate step size that causes Euler to fail but RK4 handles

t_eu, y_eu = euler(sho, 0, y0_sho, 50, h)
t_rk, y_rk = rk4(sho,  0, y0_sho, 50, h)

E_eu = 0.5 * y_eu[1]**2 + 0.5 * omega**2 * y_eu[0]**2
E_rk = 0.5 * y_rk[1]**2 + 0.5 * omega**2 * y_rk[0]**2

fig, axes = plt.subplots(1, 2, figsize=(15, 4))

# Phase portraits
ax = axes[0]
t_exact = np.linspace(0, 50, 5000)
ax.plot(np.cos(t_exact), -np.sin(t_exact), 'k-', lw=2, label='Exact', alpha=0.5)
ax.plot(y_eu[0], y_eu[1], 'r-', lw=1.2, alpha=0.8, label=f'Euler h={h}')
ax.plot(y_rk[0], y_rk[1], 'b-', lw=1.2, alpha=0.8, label=f'RK4  h={h}')
ax.set_title('Phase portrait', fontsize=12)
ax.set_xlabel('$x$'); ax.set_ylabel('$v$')
ax.legend(fontsize=9); ax.grid(alpha=0.3); ax.set_aspect('equal')

# Energy
ax = axes[1]
ax.plot(t_eu, E_eu, 'r-', lw=1.5, label=f'Euler (h={h})')
ax.plot(t_rk, E_rk, 'b-', lw=1.5, label=f'RK4   (h={h})')
ax.axhline(0.5, color='k', ls='--', lw=1.5, label='True energy')
ax.set_title('Energy vs time', fontsize=12)
ax.set_xlabel('$t$'); ax.set_ylabel('$E(t)$')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

This last result shows the difference in behaviour between Euler and RK4 for this choice of $h$. In the phase plot, the Euler method spirals out (as we've seen before), whereas RK4 stays close to the exact circle.

When looking at the plot of energy vs time, RK4 maintains nearly constant energy for far longer.

## 5. Using `scipy.integrate.solve_ivp`

As we have seen in previous notebooks, while it can be instructive to see how particular methods implemented, the really useful ones are already implemented for you in library routines. For solving ODEs (or 'initial value problems', IVP), in practice, you should use `scipy.integrate.solve_ivp`. It
implements adaptive step-size control (automatically choosing $h$ to meet a specified
tolerance), can handle stiff equations, and provides rich output.

However, understanding the underlying methods (as you now do) is essential for:
- Debugging unexpected results
- Choosing appropriate tolerances and methods
- Recognising when a solver is struggling (stiffness, singularities)

In [ ]:
# ── Using scipy for comparison ────────────────────────────────────────────────
from scipy.integrate import solve_ivp

# solve_ivp with the default RK45 method (adaptive Runge-Kutta 4/5)
sol = solve_ivp(sho, t_span=(0, 50), y0=[1.0, 0.0],
                method='RK45', rtol=1e-8, atol=1e-10) 

# the major difference with the output of solve_ivp is that the bits of the solution you need are attributes of the returned object: i.e. we access them with sol.t or sol.y:
t_sp = sol.t
y_sp = sol.y
# sol.y is the solution vector, i.e. sol.y[0] contains the values of y sol.y[1] contains the values of dy/dt. 

E_sp = 0.5 * y_sp[1]**2 + 0.5 * omega**2 * y_sp[0]**2

print("scipy solve_ivp with RK45 (rtol=1e-8, atol=1e-10):")
print(f"  Number of function evaluations: {sol.nfev}")
print(f"  Energy range: [{E_sp.min():.8f}, {E_sp.max():.8f}]  (true energy = 0.5)")
print(f"  Energy variation: {(E_sp.max() - E_sp.min()):.2e}")
print()

<div style="background:#f7d6e3;border-left:4px solid #f32121;padding:16px;margin:16px 0;border-radius:4px;width:100%;box-sizing:border-box;overflow-wrap:break-word;">

# Exercises

**1. Pendulum.** The nonlinear pendulum equation is $\ddot{\theta} = -(g/L)\sin\theta$.
Compare small-angle ($\sin\theta \approx \theta$) and full nonlinear solutions using RK4
for a pendulum _released from rest_ at initial angles $\theta_0 = 0.1, 0.5, 1.0, 1.5$ radians. 

To do this
1. Define a function `pendulum` which takes input arguments `t` and `state` where `state` is a list containing the values for the angle $\theta$ and the angular velocity $\dot{\theta}$. The function should return the values of $\frac{d}{dt}\theta$ and $\frac{d}{dt}\dot{\theta}$ as calculated using $\ddot{\theta} = -(g/L)\sin\theta$.
2. Define a function `simple` which takes input arguments `t` and `state` where `state` is a list containing the values for the angle $\theta$ and the angular velocity $\dot{\theta}$. The function should return the values of $\frac{d}{dt}\theta$ and $\frac{d}{dt}\dot{\theta}$ as calculated using $\ddot{\theta} = -(g/L)\theta$.

Then you may use `solve_ivp` with your two functions, and the various initial conditions, and create plots to compare when the small angle approximation is valid. 

Use `g=9.8` and `L=1` as defined already in the code box for the purposes of the testing. 

<div style="background:#e8f4f8;border-left:4px solid #2196F3;padding:16px;margin:16px 0;border-radius:4px;width:100%;box-sizing:border-box;overflow-wrap:break-word;">

**You might also try to extract the time period calculated by each of your methods by looking for the values of `t` when the position is zero (or one). This isn't as simple as using `if sol.y == 0`. Can you work out why?**

<details style="margin: 1em 0;">
<summary style="cursor:pointer; font-weight:bold; color:#2196F3;">
 Reveal answer
</summary>
<div style="background:#f0f7ff; border-left:4px solid #2196F3; padding:12px 16px; margin-top:8px; border-radius:4px; box-sizing:border-box; overflow-wrap:break-word;">

You might try something like

```python3
zero_t = []
for tt, yy in zip(sol.t, sol.y):
    if yy == 0:
        zero_t.append(tt)
```
Remember, we have a set of 'coordinate pairs' $(t, y)$ and we're trying to find the values of $t$ where $y=0$, so the logic of this loop is that we run through all the pairs, and make a list of all the values of $t$ that are paired with $y=0$. However, if you do this, you'll get an empty list. The reason is that we are working with floating point (decimal) numbers, and zero is never exactly zero. Something like `np.sin(np.pi)` doesn't evaluate to `0` in python, even though $\sin(\pi) \equiv 0$. Instead it will give a really small number like `1.2246467991473532e-16`. If you did the exercises on root finding, and/or the exercises on fitting you might know you can do something like this instead:

```python3
from scipy.interpolate import UnivariateSpline

spline = UnivariateSpline(sol.t, sol.y, s=0)
print(spline.roots())
```
which fits the solution of the differential equation with a cubic spline (with zero smoothing `s=0`) and then uses the resulting function (not list of numbers) to find the roots. 
</div>
</details>

</div>

</div>

In [ ]:
g = 9.8
L = 1

# your code goes here

In [ ]:
runtest(['test_pendulum', 'test_simple'])

<div style="background:#f7d6e3;border-left:4px solid #f32121;padding:16px;margin:16px 0;border-radius:4px;width:100%;box-sizing:border-box;overflow-wrap:break-word;">

# Exercise

**2. Stiffness.** The equation $dy/dt = -1000y + 3000 - 2000e^{-t}$ is stiff: this means that there are several components in the solution that vary at very different rates -- the exponential term decays quickly on a timescale of $10^{-3}$ while the linear term means solution varies on timescale $O(1)$. Solving an equation of this type puts a lot of strain on methods like RK4. 

1. Verify this difficulty by using the 'manual' `rk4` function we defined earlier for different values of the step size, to show that you need a step size of $h\lesssim 0.002$ for a stable solution. Define a function `stiff` which takes `t` and `state=[y]` and returns the value of $dy/dt$, and use this as the function you pass to `rk4`.
2. Now, instead of using `rk4` use `scipy.integrate.solve_ivp` with the method `'RK45'` and then again with the method `'Radau'`. The Radau method is specialised for stiff equations. Store the solution you get from each in `sol_rk4` and `sol_rad` respectively. To see how hard each method has to work, check the number of 'function evaluations', which you can get with 
```python
sol_rk4 = solve_ivp(...)
print(sol_rk4.nfev)
```
You should observe that the Radau solver requires far fewer function evaluations.

</div>

In [ ]:
# your code goes here

In [ ]:
runtest(['test_stiff'])

<div style="background:#f7d6e3;border-left:4px solid #f32121;padding:16px;margin:16px 0;border-radius:4px;width:100%;box-sizing:border-box;overflow-wrap:break-word;">

# Exercise


**3. Lorenz system.** The Lorenz system is:
$$\dot{x} = \sigma(y-x), \quad \dot{y} = x(\rho - z) - y, \quad \dot{z} = xy - \beta z$$
with $\sigma=10, \rho=28, \beta=8/3$. Integrate using RK4 from $t=0$ to $t=40$
and plot the solution in 3D. (Note, to plot in 3D use:

```python

ax = plt.figure().add_subplot(projection='3d')
ax.plot(x, y, z) 
```
where `x, y, z` are arrays containing the $x$, $y$ and $z$ coordinates you get as the solution of the differential equation.

<div style="background:#e8f4f8;border-left:4px solid #2196F3;padding:16px;margin:16px 0;border-radius:4px;width:100%;box-sizing:border-box;overflow-wrap:break-word;">

**What do you notice if you find the solution for two very slightly different initial conditions, e.g. `y0_1=[1, 1, 1]` and `y0_2=[1.0000001, 1, 1]`?**

<details style="margin: 1em 0;">
<summary style="cursor:pointer; font-weight:bold; color:#2196F3;">
 Reveal answer
</summary>
<div style="background:#f0f7ff; border-left:4px solid #2196F3; padding:12px 16px; margin-top:8px; border-radius:4px; box-sizing:border-box; overflow-wrap:break-word;">

Even though the initial conditions differ by only a tiny amount, and even though the solution looks 'broadly' similar when plotted in 3D, if you look closer, you'll see that the solutions are drastically different. E.G, plot just the $x$ component of the solutions with 

```python3

plt.plot(sol_1.t, sol_1.y[0])
plt.plot(sol_2.t, sol_2.y[0])
```
at some point they diverge completely from one another. This is the definition of deterministic chaos: when even a very slight difference in initial conditions leads to a huge difference in the solution.
</div>
</details>

</div>

</div>

In [ ]:
omega = 10
rho = 28
beta = 8/3

# your code goes here

In [ ]:
runtest(['test_butterfly'])